In [0]:
# Databricks notebook source
# ETL - Squad 3 - ecommerce_rastreamento_entregas
# Camada Gold
# Notebook: SLA mensal de entregas por transportadora
# Fluxo: Silver Delta -> Gold Delta -> SQL Server

In [0]:
%run "../config/00_config"

In [0]:
%run "../utils/00_utils"

In [0]:
# Define origem Silver, destino Gold, chave mensal por transportadora e tabelas SQL.

SILVER_TABLE = "ecommerce_rastreamento_entregas"
GOLD_TABLE = "gold_ecommerce_rastreamento_entregas_sla_mensal"

SILVER_PATH = f"{SILVER_BASE_PATH}{SILVER_TABLE}"
GOLD_PATH = f"{GOLD_BASE_PATH}{GOLD_TABLE}"

GOLD_KEY_COLUMNS = [
    "ano_entrega",
    "mes_entrega",
    "id_transportadora"
]

SQL_FINAL_TABLE = f"{TARGET_SCHEMA}.{GOLD_TABLE}"
SQL_STAGING_TABLE = f"{TARGET_SCHEMA}.stg_{GOLD_TABLE}"

print("SILVER_PATH:", SILVER_PATH)
print("GOLD_PATH:", GOLD_PATH)
print("GOLD_KEY_COLUMNS:", GOLD_KEY_COLUMNS)
print("SQL_FINAL_TABLE:", SQL_FINAL_TABLE)
print("SQL_STAGING_TABLE:", SQL_STAGING_TABLE)

In [0]:
# Recupera opções de conexão com o ADLS e lê a Silver de rastreamento.

adls_options = get_adls_options()

df_silver = (
    spark.read
    .format("delta")
    .options(**adls_options)
    .load(SILVER_PATH)
)

df_silver.printSchema()
display(df_silver.limit(10))

In [0]:
# Valida colunas mínimas exigidas para calcular SLA mensal de entregas.

required_columns = [
    "id_rastreamento",
    "id_pedido_ecommerce",
    "id_transportadora",
    "status_entrega",
    "dt_evento",
    "dias_em_transito"
]

validate_required_columns(df_silver, required_columns)

print("Colunas obrigatórias validadas com sucesso.")

In [0]:
# Confere se a Silver possui registros para processamento.

total_silver = df_silver.count()

print("Total de registros na Silver:", total_silver)

if total_silver == 0:
    raise Exception("A Silver está vazia. Não é possível criar a Gold.")

In [0]:
# Verifica os status disponíveis antes de filtrar apenas entregas concluídas.

from pyspark.sql.functions import col, count

df_silver.groupBy("status_entrega").agg(
    count("*").alias("qtd_registros")
).orderBy("status_entrega").show(truncate=False)

In [0]:
# Importa funções Spark usadas na regra de negócio da KPI SLA.

from pyspark.sql.functions import (
    col,
    year,
    month,
    lit,
    countDistinct,
    sum as spark_sum,
    when,
    round as spark_round,
    current_timestamp,
    to_date,
    concat_ws,
    lpad,
    count
)

from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

In [0]:
# Define o SLA prometido usado na regra de negócio.

SLA_PROMETIDO_DIAS = 7

print("SLA prometido considerado:", SLA_PROMETIDO_DIAS, "dias")

In [0]:
# Filtra registros elegíveis para a KPI:
# apenas entregas concluídas e com campos essenciais preenchidos.

df_entregas = (
    df_silver
    .filter(col("status_entrega") == "entregue")
    .filter(col("id_pedido_ecommerce").isNotNull())
    .filter(col("id_transportadora").isNotNull())
    .filter(col("dt_evento").isNotNull())
    .filter(col("dias_em_transito").isNotNull())
)

total_entregas = df_entregas.count()

print("Total de eventos com status entregue:", total_entregas)

if total_entregas == 0:
    raise Exception("Não existem registros com status_entrega = 'entregue'. Não é possível calcular SLA.")

In [0]:
# Mantém uma única entrega por pedido.
# Isso evita contar o mesmo pedido mais de uma vez na KPI.

window_pedido_entregue = (
    Window
    .partitionBy("id_pedido_ecommerce")
    .orderBy(col("dt_evento").desc(), col("id_rastreamento").desc())
)

df_entregas_unicas = (
    df_entregas
    .withColumn("rn", row_number().over(window_pedido_entregue))
    .filter(col("rn") == 1)
    .drop("rn")
)

print("Total de pedidos entregues únicos:", df_entregas_unicas.count())

display(
    df_entregas_unicas
    .select(
        "id_pedido_ecommerce",
        "id_rastreamento",
        "id_transportadora",
        "status_entrega",
        "dt_evento",
        "dias_em_transito"
    )
    .orderBy("dt_evento")
    .limit(20)
)

In [0]:
# Aplica a regra de negócio da KPI SLA:
# dentro do SLA = dias_em_transito <= 7

df_gold = (
    df_entregas_unicas
    .withColumn("ano_entrega", year(col("dt_evento")))
    .withColumn("mes_entrega", month(col("dt_evento")))
    .withColumn(
        "fl_dentro_sla",
        when(col("dias_em_transito") <= SLA_PROMETIDO_DIAS, 1).otherwise(0)
    )
    .withColumn(
        "fl_fora_sla",
        when(col("dias_em_transito") > SLA_PROMETIDO_DIAS, 1).otherwise(0)
    )
    .groupBy(
        "ano_entrega",
        "mes_entrega",
        "id_transportadora"
    )
    .agg(
        countDistinct("id_pedido_ecommerce").alias("qtd_pedidos_entregues"),
        spark_sum("fl_dentro_sla").alias("qtd_pedidos_dentro_sla"),
        spark_sum("fl_fora_sla").alias("qtd_pedidos_fora_sla")
    )
    .withColumn(
        "percentual_dentro_sla",
        spark_round(
            (col("qtd_pedidos_dentro_sla") / col("qtd_pedidos_entregues")) * 100,
            2
        )
    )
    .withColumn(
        "percentual_fora_sla",
        spark_round(
            (col("qtd_pedidos_fora_sla") / col("qtd_pedidos_entregues")) * 100,
            2
        )
    )
    .withColumn("sla_prometido_dias", lit(SLA_PROMETIDO_DIAS))
    .withColumn(
        "data_referencia",
        to_date(
            concat_ws(
                "-",
                col("ano_entrega"),
                lpad(col("mes_entrega"), 2, "0"),
                lit("01")
            )
        )
    )
    .withColumn("gold_processed_at", current_timestamp())
    .select(
        "ano_entrega",
        "mes_entrega",
        "data_referencia",
        "id_transportadora",
        "qtd_pedidos_entregues",
        "qtd_pedidos_dentro_sla",
        "qtd_pedidos_fora_sla",
        "percentual_dentro_sla",
        "percentual_fora_sla",
        "sla_prometido_dias",
        "gold_processed_at"
    )
    .orderBy(
        "ano_entrega",
        "mes_entrega",
        "id_transportadora"
    )
)

display(df_gold)

In [0]:
# Valida se a Gold em memória foi gerada com registros.

total_gold = df_gold.count()

print("Total de linhas na Gold em memória:", total_gold)

if total_gold == 0:
    raise Exception("A Gold em memória está vazia.")

In [0]:
# Valida a chave lógica da Gold para evitar duplicidade por mês/transportadora.

df_gold_duplicadas = (
    df_gold
    .groupBy(GOLD_KEY_COLUMNS)
    .agg(count("*").alias("qtd_linhas"))
    .filter(col("qtd_linhas") > 1)
)

qtd_duplicadas = df_gold_duplicadas.count()

print("Quantidade de chaves duplicadas:", qtd_duplicadas)

if qtd_duplicadas > 0:
    display(df_gold_duplicadas)
    raise Exception("Existem chaves duplicadas na Gold.")

print("Validação OK: não existem chaves duplicadas.")

In [0]:
# Valida campos obrigatórios que não podem ficar nulos na Gold.

df_gold_nulos = (
    df_gold
    .filter(
        col("ano_entrega").isNull() |
        col("mes_entrega").isNull() |
        col("data_referencia").isNull() |
        col("id_transportadora").isNull() |
        col("qtd_pedidos_entregues").isNull() |
        col("qtd_pedidos_dentro_sla").isNull() |
        col("qtd_pedidos_fora_sla").isNull() |
        col("percentual_dentro_sla").isNull() |
        col("percentual_fora_sla").isNull() |
        col("sla_prometido_dias").isNull()
    )
)

qtd_nulos = df_gold_nulos.count()

print("Quantidade de linhas com campos obrigatórios nulos:", qtd_nulos)

if qtd_nulos > 0:
    display(df_gold_nulos)
    raise Exception("Existem campos obrigatórios nulos na Gold.")

print("Validação OK: campos obrigatórios sem nulos.")

In [0]:
# Valida se a soma dentro/fora do SLA bate com o total de pedidos entregues.

df_gold_validacao_sla = (
    df_gold
    .withColumn(
        "qtd_check",
        col("qtd_pedidos_dentro_sla") + col("qtd_pedidos_fora_sla")
    )
    .withColumn(
        "diferenca",
        col("qtd_pedidos_entregues") - col("qtd_check")
    )
    .filter(col("diferenca") != 0)
)

qtd_erros_sla = df_gold_validacao_sla.count()

print("Linhas com divergência no cálculo de SLA:", qtd_erros_sla)

if qtd_erros_sla > 0:
    display(df_gold_validacao_sla)
    raise Exception("Erro: dentro_sla + fora_sla não bate com o total de pedidos entregues.")

print("Validação OK: dentro_sla + fora_sla bate com o total de pedidos entregues.")

In [0]:
# Confere se o total de pedidos entregues da Gold bate com a origem filtrada.

total_pedidos_entregues_origem = (
    df_entregas_unicas
    .select("id_pedido_ecommerce")
    .distinct()
    .count()
)

total_pedidos_entregues_gold = (
    df_gold
    .agg(spark_sum("qtd_pedidos_entregues").alias("total"))
    .collect()[0]["total"]
)

print("Total de pedidos entregues únicos na origem:", total_pedidos_entregues_origem)
print("Total de pedidos entregues somados na Gold:", total_pedidos_entregues_gold)

if total_pedidos_entregues_origem != total_pedidos_entregues_gold:
    raise Exception("A soma de pedidos da Gold não bate com os pedidos entregues da origem.")

print("Validação OK: total de pedidos entregues confere.")

In [0]:
# Exibe schema e amostra final da Gold antes da gravação.

df_gold.printSchema()

display(
    df_gold
    .orderBy(
        "ano_entrega",
        "mes_entrega",
        "id_transportadora"
    )
)

In [0]:
# Valida nome, caminho e chave lógica antes da escrita Delta.

print("Tabela Gold:", GOLD_TABLE)
print("Caminho Gold:", GOLD_PATH)
print("Chave lógica:", GOLD_KEY_COLUMNS)

if not GOLD_TABLE.startswith("gold_"):
    raise Exception("Nome da tabela Gold fora do padrão esperado.")

print("Validações iniciais da escrita Delta concluídas.")

In [0]:
# Verifica se a Gold Delta já existe no ADLS.

try:
    df_gold_existente = (
        spark.read
        .format("delta")
        .options(**adls_options)
        .load(GOLD_PATH)
    )
    
    gold_existe = True
    print("Gold Delta existente encontrada.")
    print("Linhas atuais na Gold Delta:", df_gold_existente.count())

except Exception as e:
    gold_existe = False
    df_gold_existente = None
    print("Gold Delta ainda não existe. Será criada na primeira gravação.")

In [0]:
# Monta o upsert manual, substituindo apenas as chaves recalculadas.

df_gold_novas_chaves = df_gold.select(GOLD_KEY_COLUMNS).distinct()

if gold_existe:
    df_gold_existente_restante = (
        df_gold_existente
        .join(
            df_gold_novas_chaves,
            on=GOLD_KEY_COLUMNS,
            how="left_anti"
        )
    )

    df_gold_final = (
        df_gold_existente_restante
        .unionByName(df_gold, allowMissingColumns=True)
    )

else:
    df_gold_final = df_gold

print("Linhas que serão mantidas/gravadas na Gold Delta:", df_gold_final.count())

In [0]:
# Grava a Gold Delta particionada por ano e mês de entrega.

(
    df_gold_final
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .options(**adls_options)
    .partitionBy("ano_entrega", "mes_entrega")
    .save(GOLD_PATH)
)

print("Gold Delta salva com sucesso.")

In [0]:
# Lê novamente a Gold Delta gravada para validação pós-escrita.

df_gold_delta = (
    spark.read
    .format("delta")
    .options(**adls_options)
    .load(GOLD_PATH)
)

print("Total de linhas gravadas na Gold Delta:", df_gold_delta.count())

display(
    df_gold_delta
    .orderBy(
        "ano_entrega",
        "mes_entrega",
        "id_transportadora"
    )
)

In [0]:
# Confere se a quantidade gravada bate com o DataFrame final em memória.

total_gold_final_memoria = df_gold_final.count()
total_gold_delta = df_gold_delta.count()

print("Total esperado na Gold Delta:", total_gold_final_memoria)
print("Total lido da Gold Delta:", total_gold_delta)

if total_gold_final_memoria != total_gold_delta:
    raise Exception("A quantidade de linhas da Gold Delta não bate com o DataFrame final em memória.")

print("Quantidade de linhas validada com sucesso.")

In [0]:
# Valida se a Gold Delta não possui duplicidade por mês/transportadora.

df_gold_delta_duplicadas = (
    df_gold_delta
    .groupBy(GOLD_KEY_COLUMNS)
    .agg(count("*").alias("qtd_linhas"))
    .filter(col("qtd_linhas") > 1)
)

qtd_duplicadas_delta = df_gold_delta_duplicadas.count()

print("Quantidade de chaves duplicadas na Gold Delta:", qtd_duplicadas_delta)

if qtd_duplicadas_delta > 0:
    display(df_gold_delta_duplicadas)
    raise Exception("Existem chaves duplicadas na Gold Delta.")

print("Duplicidade de chave validada com sucesso.")

In [0]:
# Valida se todas as colunas finais da Gold Delta estão presentes.

required_gold_columns = [
    "ano_entrega",
    "mes_entrega",
    "data_referencia",
    "id_transportadora",
    "qtd_pedidos_entregues",
    "qtd_pedidos_dentro_sla",
    "qtd_pedidos_fora_sla",
    "percentual_dentro_sla",
    "percentual_fora_sla",
    "sla_prometido_dias",
    "gold_processed_at"
]

validate_required_columns(df_gold_delta, required_gold_columns)

print("Colunas obrigatórias da Gold Delta validadas com sucesso.")

In [0]:
# Confere se todas as chaves da execução atual foram gravadas no Delta.

df_chaves_nao_gravadas = (
    df_gold
    .select(GOLD_KEY_COLUMNS)
    .distinct()
    .join(
        df_gold_delta.select(GOLD_KEY_COLUMNS).distinct(),
        on=GOLD_KEY_COLUMNS,
        how="left_anti"
    )
)

qtd_chaves_nao_gravadas = df_chaves_nao_gravadas.count()

print("Quantidade de chaves da execução atual não encontradas na Gold Delta:", qtd_chaves_nao_gravadas)

if qtd_chaves_nao_gravadas > 0:
    display(df_chaves_nao_gravadas)
    raise Exception("Algumas chaves da execução atual não foram gravadas na Gold Delta.")

print("Chaves da execução atual validadas com sucesso.")

In [0]:
# Valida schema e nome da staging antes da escrita no SQL Server.

print("Schema alvo:", TARGET_SCHEMA)
print("Tabela staging:", SQL_STAGING_TABLE)

if TARGET_SCHEMA != "squad3":
    raise Exception("Schema alvo diferente de squad3. Escrita bloqueada por segurança.")

if not SQL_STAGING_TABLE.startswith("squad3.stg_"):
    raise Exception("Tabela staging fora do padrão esperado. Escrita bloqueada por segurança.")

print("Validações de segurança para staging concluídas.")

In [0]:
# Seleciona as colunas finais que serão enviadas para a staging SQL.

df_sql_staging = df_gold_delta.select(
    "ano_entrega",
    "mes_entrega",
    "data_referencia",
    "id_transportadora",
    "qtd_pedidos_entregues",
    "qtd_pedidos_dentro_sla",
    "qtd_pedidos_fora_sla",
    "percentual_dentro_sla",
    "percentual_fora_sla",
    "sla_prometido_dias",
    "gold_processed_at"
)

display(df_sql_staging)

In [0]:
# Grava a staging no SQL Server.

write_sql_table(
    df=df_sql_staging,
    table_name=SQL_STAGING_TABLE,
    sql_host=SQL_HOST,
    sql_database=SQL_DATABASE,
    sql_username=SQL_USERNAME,
    sql_password=SQL_PASSWORD,
    mode="overwrite"
)

print("Tabela staging gravada no SQL Server com sucesso.")

In [0]:
# Lê a staging gravada para validar a escrita no SQL Server.

df_sql_staging_lida = read_sql_table(
    spark=spark,
    table_name=SQL_STAGING_TABLE,
    sql_host=SQL_HOST,
    sql_database=SQL_DATABASE,
    sql_username=SQL_USERNAME,
    sql_password=SQL_PASSWORD
)

print("Total de linhas lidas da staging:", df_sql_staging_lida.count())

display(
    df_sql_staging_lida
    .orderBy(
        "ano_entrega",
        "mes_entrega",
        "id_transportadora"
    )
)

In [0]:
# Confere se a staging possui a mesma quantidade de linhas esperada.

total_staging_esperado = df_sql_staging.count()
total_staging_lido = df_sql_staging_lida.count()

print("Total esperado na staging:", total_staging_esperado)
print("Total lido da staging:", total_staging_lido)

if total_staging_esperado != total_staging_lido:
    raise Exception("A quantidade de linhas da staging não bate com a Gold Delta.")

print("Staging SQL Server validada com sucesso.")

In [0]:
# Valida schema e nome da tabela final antes da publicação no SQL Server.

print("Schema alvo:", TARGET_SCHEMA)
print("Tabela final:", SQL_FINAL_TABLE)

if TARGET_SCHEMA != "squad3":
    raise Exception("Schema alvo diferente de squad3. Escrita bloqueada por segurança.")

if not SQL_FINAL_TABLE.startswith("squad3.gold_"):
    raise Exception("Tabela final fora do padrão esperado. Escrita bloqueada por segurança.")

print("Validações de segurança para tabela final concluídas.")

In [0]:
# Verifica se a tabela final já existe no SQL Server.

try:
    df_sql_final_existente = read_sql_table(
        spark=spark,
        table_name=SQL_FINAL_TABLE,
        sql_host=SQL_HOST,
        sql_database=SQL_DATABASE,
        sql_username=SQL_USERNAME,
        sql_password=SQL_PASSWORD
    )

    sql_final_existe = True
    print("Tabela final existente encontrada.")
    print("Linhas atuais na tabela final:", df_sql_final_existente.count())

except Exception as e:
    sql_final_existe = False
    df_sql_final_existente = None
    print("Tabela final ainda não existe. Será criada na primeira gravação.")

In [0]:
# Monta o upsert manual da tabela final com base na chave da Gold.

df_sql_novas_chaves = df_sql_staging_lida.select(GOLD_KEY_COLUMNS).distinct()

if sql_final_existe:
    df_sql_final_restante = (
        df_sql_final_existente
        .join(
            df_sql_novas_chaves,
            on=GOLD_KEY_COLUMNS,
            how="left_anti"
        )
    )

    df_sql_final = (
        df_sql_final_restante
        .unionByName(df_sql_staging_lida, allowMissingColumns=True)
    )

else:
    df_sql_final = df_sql_staging_lida

print("Linhas que serão gravadas na tabela final:", df_sql_final.count())

display(
    df_sql_final
    .orderBy(
        "ano_entrega",
        "mes_entrega",
        "id_transportadora"
    )
)

In [0]:
# Grava ou atualiza a tabela final no SQL Server.

write_sql_table(
    df=df_sql_final,
    table_name=SQL_FINAL_TABLE,
    sql_host=SQL_HOST,
    sql_database=SQL_DATABASE,
    sql_username=SQL_USERNAME,
    sql_password=SQL_PASSWORD,
    mode="overwrite"
)

print("Tabela final gravada/atualizada no SQL Server com sucesso.")

In [0]:
# Valida quantidade e duplicidade da tabela final publicada.

df_sql_final_lida = read_sql_table(
    spark=spark,
    table_name=SQL_FINAL_TABLE,
    sql_host=SQL_HOST,
    sql_database=SQL_DATABASE,
    sql_username=SQL_USERNAME,
    sql_password=SQL_PASSWORD
)

total_final_esperado = df_sql_final.count()
total_final_lido = df_sql_final_lida.count()

print("Total esperado na tabela final:", total_final_esperado)
print("Total lido da tabela final:", total_final_lido)

if total_final_esperado != total_final_lido:
    raise Exception("A quantidade de linhas da tabela final não bate com o esperado.")

df_sql_final_duplicadas = (
    df_sql_final_lida
    .groupBy(GOLD_KEY_COLUMNS)
    .agg(count("*").alias("qtd_linhas"))
    .filter(col("qtd_linhas") > 1)
)

qtd_duplicadas_final = df_sql_final_duplicadas.count()

print("Quantidade de chaves duplicadas na tabela final:", qtd_duplicadas_final)

if qtd_duplicadas_final > 0:
    display(df_sql_final_duplicadas)
    raise Exception("Existem chaves duplicadas na tabela final.")

display(
    df_sql_final_lida
    .orderBy(
        "ano_entrega",
        "mes_entrega",
        "id_transportadora"
    )
)

print("Tabela final SQL Server validada com sucesso.")

In [0]:
# Exibe o resumo final da execução da Gold SLA mensal de entregas.

print("Resumo da execução")
print("-" * 50)

print("Notebook:")
print("03_gold_squad3_ecommerce_rastreamento_entregas_sla_mensal")

print("\nTabela Gold Delta:")
print(GOLD_TABLE)

print("\nCaminho Gold Delta:")
print(GOLD_PATH)

print("\nTabela staging SQL Server:")
print(SQL_STAGING_TABLE)

print("\nTabela final SQL Server:")
print(SQL_FINAL_TABLE)

print("\nChave lógica:")
print(GOLD_KEY_COLUMNS)

print("\nSLA prometido considerado:")
print(f"{SLA_PROMETIDO_DIAS} dias")

print("\nTotal de linhas na Gold Delta:")
print(df_gold_delta.count())

print("\nTotal de linhas na tabela final SQL Server:")
print(df_sql_final_lida.count())

print("\nStatus:")
print("Gold de SLA mensal de entregas por transportadora concluída com sucesso.")

In [0]:
# Define as Golds de rastreamento que serão revisadas.

gold_tables_review = [
    {
        "nome": "tempo_medio_mensal",
        "gold_table": "gold_ecommerce_rastreamento_entregas_tempo_medio_mensal",
        "key_columns": ["ano_entrega", "mes_entrega", "id_transportadora"]
    },
    {
        "nome": "sla_mensal",
        "gold_table": "gold_ecommerce_rastreamento_entregas_sla_mensal",
        "key_columns": ["ano_entrega", "mes_entrega", "id_transportadora"]
    },
    {
        "nome": "problemas_mensal",
        "gold_table": "gold_ecommerce_rastreamento_entregas_problemas_mensal",
        "key_columns": ["ano_evento", "mes_evento", "id_transportadora"]
    }
]

adls_options = get_adls_options()

print("Golds que serão revisadas:")

for item in gold_tables_review:
    print("-", item["gold_table"], "| chave:", item["key_columns"])

In [0]:
# Revisa existência, quantidade de linhas e duplicidade de chave nas Golds Delta.

from pyspark.sql.functions import count

review_results = []

for item in gold_tables_review:
    gold_table = item["gold_table"]
    key_columns = item["key_columns"]
    gold_path = f"{GOLD_BASE_PATH}{gold_table}"

    print("-" * 80)
    print("Revisando Gold Delta:", gold_table)
    print("Caminho:", gold_path)

    df_delta = (
        spark.read
        .format("delta")
        .options(**adls_options)
        .load(gold_path)
    )

    total_linhas = df_delta.count()

    qtd_duplicadas = (
        df_delta
        .groupBy(key_columns)
        .agg(count("*").alias("qtd_linhas"))
        .filter(col("qtd_linhas") > 1)
        .count()
    )

    review_results.append(
        {
            "camada": "delta",
            "gold_table": gold_table,
            "total_linhas": total_linhas,
            "qtd_chaves_duplicadas": qtd_duplicadas,
            "status": "OK" if qtd_duplicadas == 0 else "ERRO"
        }
    )

    print("Total de linhas:", total_linhas)
    print("Chaves duplicadas:", qtd_duplicadas)

    if qtd_duplicadas > 0:
        raise Exception(f"Existem chaves duplicadas na Gold Delta: {gold_table}")

print("Revisão das Golds Delta concluída.")

In [0]:
# Revisa existência, quantidade de linhas e duplicidade de chave nas tabelas finais SQL Server.

for item in gold_tables_review:
    gold_table = item["gold_table"]
    key_columns = item["key_columns"]
    sql_final_table = f"{TARGET_SCHEMA}.{gold_table}"

    print("-" * 80)
    print("Revisando tabela final SQL:", sql_final_table)

    df_sql = read_sql_table(
        spark=spark,
        table_name=sql_final_table,
        sql_host=SQL_HOST,
        sql_database=SQL_DATABASE,
        sql_username=SQL_USERNAME,
        sql_password=SQL_PASSWORD
    )

    total_linhas = df_sql.count()

    qtd_duplicadas = (
        df_sql
        .groupBy(key_columns)
        .agg(count("*").alias("qtd_linhas"))
        .filter(col("qtd_linhas") > 1)
        .count()
    )

    review_results.append(
        {
            "camada": "sql_final",
            "gold_table": gold_table,
            "total_linhas": total_linhas,
            "qtd_chaves_duplicadas": qtd_duplicadas,
            "status": "OK" if qtd_duplicadas == 0 else "ERRO"
        }
    )

    print("Total de linhas:", total_linhas)
    print("Chaves duplicadas:", qtd_duplicadas)

    if qtd_duplicadas > 0:
        raise Exception(f"Existem chaves duplicadas na tabela final SQL: {sql_final_table}")

print("Revisão das tabelas finais SQL concluída.")

In [0]:
# Exibe resumo consolidado da revisão.

df_review = spark.createDataFrame(review_results)

display(
    df_review
    .orderBy("gold_table", "camada")
)

print("Resumo da revisão concluído.")